In [14]:
import pandas as pd
import numpy as np
import tqdm as notebook_tqdm
import ast
import re
from transformers import pipeline
import json
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout

In [2]:
emotion_analyzer = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")

emotion_map = {
    "anger": 0, "disgust": 1, "fear": 2,
    "joy": 3, "neutral": 4, "sadness": 5, "surprise": 6
}

def analyze_emotion(text):
    return emotion_analyzer(text)

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
def prev_k(p1, p2, k=1):
    X = []
    y = []
    context = np.zeros((k,len(emotion_map)))
    for i in range(k):
        context[i] = analyze_emotion(p1[i])
    X.append(context)
    y.append(analyze_emotion(p2[k-1]))
    for i in range(k,len(p1)):
        context = context[1:,:]
        np.vstack([context, analyze_emotion(p1[i])])
        X.append(context)
        y.append(analyze_emotion(p2[i]))

In [3]:
lines = {}
conversations = []
p1 = []
p2 = []

def clean_text(text):
    text = re.sub(r"[\[\]]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace('&amp;','and')
    return text.strip()

with open('/Users/ashvinsehgal/AI-ML/Emotions/Data/movie/movie_conversations.txt','r',encoding='iso-8859-1') as f:
    for row in f:
        parts = row.strip().split(' +++$+++ ')
        conversations.append({
            'char1': parts[0],
            'char2': parts[1],
            'movie_id': parts[2],
            'line_ids': ast.literal_eval(parts[3])
        })
    
with open('/Users/ashvinsehgal/AI-ML/Emotions/Data/movie/movie_lines.txt','r',encoding='iso-8859-1') as f:
    for row in f:
        parts = row.strip().split(' +++$+++ ')
        if len(parts) == 5:
            lines[parts[0]] = {
                'char_id': parts[1],
                'char_name': parts[2],
                'movie_id': parts[3],
                'text': clean_text(parts[4])
            }
        
def get_conversation(conv):
    convo = []
    for line_id in conv['line_ids']:
        if line_id not in lines:
            return None
        dialog = lines[line_id]
        convo.append({
            'line_id': line_id,
            'speaker': dialog['char_id'],
            'dialog': dialog['text']
        })
    return convo

def check_1_1(convo):
    if len(convo) < 4:
        return False
    speakers = [dialog['speaker'] for dialog in convo]
    if len(set(speakers)) != 2:
        return False
    for i in range(1, len(speakers)):
        if speakers[i] == speakers[i-1]:
            return False
    return True

def get_dialogs(conversation):
    d1 = []
    d2 = []
    for i in range(0,len(conversation),2):
        d1.append(conversation[i]['dialog'])
        if i+1 < len(conversation):
            d2.append(conversation[i+1]['dialog'])
    return d1, d2
        
for conv in conversations:
    conversation = get_conversation(conv)
    if conversation and check_1_1(conversation):
        dialogs1, dialogs2 = get_dialogs(conversation)
        p1.append(dialogs1)
        p2.append(dialogs2)

In [4]:
len(p1), len(p2)

(27008, 27008)

In [16]:
df_train = pd.read_csv('/Users/ashvinsehgal/AI-ML/Emotions/Data/train.csv')
df_val = pd.read_csv('/Users/ashvinsehgal/AI-ML/Emotions/Data/validation.csv')
df_test = pd.read_csv('/Users/ashvinsehgal/AI-ML/Emotions/Data/test.csv')
emotions = {
        0: 'neutral',
        1: 'anger',
        2: 'disgust',
        3: 'fear',
        4: 'joy',
        5: 'sadness',
        6: 'surprise'
    }

conversations = []

def clean_text(text):
    text = re.sub(r"[\[\]]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace('&amp;','and')
    return text.strip()

def is_1_1(row):
    act = ast.literal_eval(row[1].replace(" ",","))
    if len(act) < 4:
        return False
    if len(set(act)) != 2:
        return False
    for i in range(1, len(act)):
        if act[i] == act[i-1]:
            return False
    return True 

def get_dialogs(conversation):
    d1 = []
    d2 = []
    for i in range(0,len(conversation),2):
        d1.append(conversation[i])
        if i+1 < len(conversation):
            d2.append(conversation[i+1])
    return d1, d2

for row in df_train.values:
    if is_1_1(row):
        conversation = ast.literal_eval(clean_text(row[0].replace("\n",",")))
        dialogs1, dialogs2 = get_dialogs(conversation)
        p1.append(dialogs1)
        p2.append(dialogs2)
for row in df_val.values:
    if is_1_1(row):
        conversation = ast.literal_eval(clean_text(row[0].replace("\n",",")))
        dialogs1, dialogs2 = get_dialogs(conversation)
        p1.append(dialogs1)
        p2.append(dialogs2)
for row in df_test.values:
    if is_1_1(row):
        conversation = ast.literal_eval(clean_text(row[0].replace("\n",",")))
        dialogs1, dialogs2 = get_dialogs(conversation)
        p1.append(dialogs1)
        p2.append(dialogs2)

In [7]:
len(p1), len(p2)

(28003, 28003)

In [8]:
conversation = []
with open('/Users/ashvinsehgal/AI-ML/Emotions/Data/human_chat.txt','r') as f:
    for row in f:
        conversation.append(clean_text(row[9:-1]))

def clean_text(text):
    text = re.sub(r"[\[\]]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace('&amp;','and')
    return text.strip()

def get_dialogs(conversation):
    d1 = []
    d2 = []
    for i in range(0,len(conversation),2):
        d1.append(conversation[i])
        if i+1 < len(conversation):
            d2.append(conversation[i+1])
    return d1, d2

dialogs1, dialogs2 = get_dialogs(conversation)
p1.append(dialogs1)
p2.append(dialogs2)

In [9]:
len(p1), len(p2)

(28004, 28004)

In [10]:
with open('p1.json','w',encoding='utf-8') as f:
    json.dump(p1,f)
with open('p2.json','w',encoding='utf-8') as f:
    json.dump(p2,f)

In [11]:
data1 = []
data2 = []
with open('/Users/ashvinsehgal/AI-ML/Emotions/Data/data1.json','r') as f:
    data1 = json.load(f)
with open('/Users/ashvinsehgal/AI-ML/Emotions/Data/data2.json','r') as f:
    data2 = json.load(f)

In [ ]:
model = Sequential([
    Dense(32),
    Dense(64),
    Dense(512),
    Dense(64),
    Dense(len(emotions))
])
model.compile(optimizer='adam',loss='')